# Notebook 27 — Paper Figures and Results Pack

Consolidates outputs from Notebooks 17–26 into a paper-ready figure/results bundle.

This notebook is intentionally tolerant: it loads existing CSV/PNG outputs when available and generates clean fallback summaries when files are missing.

Outputs:

- paper figure panels in `figures/`
- summary tables in `results/`
- caption drafts and manifest in `exports/`
- downloadable zip package


In [ ]:
# Notebook 27 setup
from pathlib import Path
import os, json, zipfile, shutil, textwrap, math, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Locate repo root robustly for Colab, local notebooks/, and GitHub checkout.
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    markers = ["notebooks", "results", ".git", "figures"]
    for c in candidates:
        if any((c / m).exists() for m in markers):
            # If running inside notebooks/, parent is usually repo root.
            if c.name == "notebooks":
                return c.parent
            return c
    return start

REPO_ROOT = find_repo_root()
# Colab often starts at /content while repo is /content/<repo>; detect a likely repo if results are empty.
if REPO_ROOT == Path('/content'):
    candidates = [p for p in Path('/content').glob('*') if p.is_dir() and (p/'notebooks').exists()]
    if candidates:
        REPO_ROOT = candidates[0]

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
EXPORTS_DIR = REPO_ROOT / "exports"
DOCS_DIR = REPO_ROOT / "docs"

for d in [RESULTS_DIR, FIGURES_DIR, EXPORTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("EXPORTS_DIR:", EXPORTS_DIR)
print("available result files:", sorted(p.name for p in RESULTS_DIR.glob('*'))[:40])
print("available figure files:", sorted(p.name for p in FIGURES_DIR.glob('*.png'))[:20])


## 1. File discovery

Collect all outputs from Notebooks 17–26. This notebook favors existing repo outputs, then falls back to generated summary material when needed.

In [ ]:
NOTEBOOK_RANGE = list(range(17, 27))

def list_numbered_files(directory, suffixes=(".csv", ".json", ".png", ".md", ".npy")):
    rows = []
    for p in sorted(directory.glob("*")):
        if not p.is_file() or p.suffix.lower() not in suffixes:
            continue
        stem = p.name
        nb = None
        for n in NOTEBOOK_RANGE:
            if stem.startswith(f"{n}_") or stem.startswith(f"0{n}_") or f"_{n}_" in stem:
                nb = n
                break
        rows.append({
            "notebook": nb,
            "name": p.name,
            "path": str(p),
            "suffix": p.suffix.lower(),
            "size_kb": round(p.stat().st_size/1024, 2),
        })
    return pd.DataFrame(rows)

result_inventory = list_numbered_files(RESULTS_DIR, suffixes=(".csv", ".json", ".npy", ".md"))
figure_inventory = list_numbered_files(FIGURES_DIR, suffixes=(".png",))

result_inventory.to_csv(RESULTS_DIR / "27_result_inventory.csv", index=False)
figure_inventory.to_csv(RESULTS_DIR / "27_figure_inventory.csv", index=False)

print("result inventory rows:", len(result_inventory))
print("figure inventory rows:", len(figure_inventory))
display(result_inventory.head(20))
display(figure_inventory.head(20))


## 2. Load core result tables

These are the most useful upstream products for a manuscript summary. Missing files are skipped cleanly.

In [ ]:
def read_csv_if_exists(name):
    p = RESULTS_DIR / name
    if p.exists():
        print("loaded", name)
        return pd.read_csv(p)
    print("missing", name)
    return None

core_files = {
    "residual_geometry_features": "residual_geometry_features.csv",
    "residual_classification_features": "residual_classification_feature_matrix.csv",
    "residual_pca_embedding": "residual_pca_embedding.csv",
    "residual_universality_embedding": "residual_universality_embedding.csv",
    "ood_transfer_assignments": "ood_transfer_assignments.csv",
    "controlled_boundary_sweeps": "controlled_boundary_sweeps.csv",
    "geodesic_transport_summary": "25_geodesic_transport_summary.csv",
    "diffusion_persistence": "26_universality_persistence.csv",
    "diffusion_entropy": "26_entropy_curves.csv",
    "diffusion_phase_grid": "26_phase_transition_grid.csv",
    "laplacian_spectrum": "26_laplacian_spectrum.csv",
}

tables = {k: read_csv_if_exists(v) for k, v in core_files.items()}
loaded_tables = {k: v for k, v in tables.items() if v is not None}
print("loaded tables:", sorted(loaded_tables))


## 3. Build compact paper-results summary

In [ ]:
summary_rows = []

# Inventory-level summaries
summary_rows.append({"section":"inventory", "metric":"result_files_17_26", "value": len(result_inventory), "note":"numbered result/output files found"})
summary_rows.append({"section":"inventory", "metric":"figure_files_17_26", "value": len(figure_inventory), "note":"numbered PNG files found"})

# Geometry summaries
geo = tables.get("residual_geometry_features")
if geo is not None:
    summary_rows.append({"section":"geometry", "metric":"geometry_rows", "value": len(geo), "note":"residual geometry feature rows"})
    for col in ["residual_entropy", "residual_asymmetry", "residual_bend_energy", "residual_localization"]:
        if col in geo.columns:
            summary_rows.append({"section":"geometry", "metric":f"mean_{col}", "value": float(pd.to_numeric(geo[col], errors='coerce').mean()), "note":"mean over available topology-size rows"})

# Classification / separability
clf = tables.get("residual_classification_features")
if clf is not None:
    summary_rows.append({"section":"separability", "metric":"classification_feature_rows", "value": len(clf), "note":"feature matrix rows"})
    if "topology" in clf.columns:
        summary_rows.append({"section":"separability", "metric":"n_topologies", "value": clf["topology"].nunique(), "note":"unique known topology labels"})

# OOD summaries
ood = tables.get("ood_transfer_assignments")
if ood is not None:
    summary_rows.append({"section":"ood", "metric":"ood_rows", "value": len(ood), "note":"OOD transfer/assignment rows"})
    for col in ["confidence", "relative_confidence", "nearest_distance", "transfer_similarity"]:
        if col in ood.columns:
            summary_rows.append({"section":"ood", "metric":f"mean_{col}", "value": float(pd.to_numeric(ood[col], errors='coerce').mean()), "note":"mean OOD transfer statistic"})

# Diffusion summaries
persist = tables.get("diffusion_persistence")
if persist is not None:
    summary_rows.append({"section":"diffusion", "metric":"persistence_rows", "value": len(persist), "note":"diffusion persistence rows"})
    for col in ["persistence", "universality_persistence", "mass_retained"]:
        if col in persist.columns:
            summary_rows.append({"section":"diffusion", "metric":f"final_mean_{col}", "value": float(persist.groupby(persist.columns[0])[col].last().mean()) if persist.columns[0] in persist.columns else float(pd.to_numeric(persist[col], errors='coerce').mean()), "note":"late/final persistence estimate"})

phase = tables.get("diffusion_phase_grid")
if phase is not None:
    summary_rows.append({"section":"diffusion", "metric":"phase_grid_rows", "value": len(phase), "note":"diffusion phase-grid rows"})
    for col in ["mean_persistence", "persistence"]:
        if col in phase.columns:
            summary_rows.append({"section":"diffusion", "metric":f"max_{col}", "value": float(pd.to_numeric(phase[col], errors='coerce').max()), "note":"strongest persistence region"})

summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "27_paper_results_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("wrote", summary_path)
display(summary_df)


## 4. Helper functions for paper figures

In [ ]:
def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print("wrote", path)
    plt.show()

def normalize_label_columns(df):
    if df is None:
        return None
    df = df.copy()
    # Normalize topology/family/name columns
    for c in ["topology", "family", "label", "known_family"]:
        if c in df.columns and "topology" not in df.columns:
            df["topology"] = df[c]
    # Normalize PC coordinates
    aliases = {
        "PC1": ["PC1", "pc1", "x", "coordinate_1", "residual manifold coordinate 1"],
        "PC2": ["PC2", "pc2", "y", "coordinate_2", "residual manifold coordinate 2"],
    }
    for target, opts in aliases.items():
        if target not in df.columns:
            for o in opts:
                if o in df.columns:
                    df[target] = df[o]
                    break
    return df

def find_first_existing_figure(keywords):
    keywords = [k.lower() for k in keywords]
    candidates = []
    for p in FIGURES_DIR.glob("*.png"):
        name = p.name.lower()
        if all(k in name for k in keywords):
            candidates.append(p)
    return sorted(candidates)[0] if candidates else None

def copy_or_placeholder(out_name, title, keywords=None):
    out = FIGURES_DIR / out_name
    src = find_first_existing_figure(keywords or []) if keywords else None
    if src and src != out:
        shutil.copy2(src, out)
        print("copied", src.name, "->", out.name)
        return out
    # placeholder
    plt.figure(figsize=(8, 4.5))
    plt.text(0.5, 0.55, title, ha='center', va='center', fontsize=18, weight='bold')
    plt.text(0.5, 0.42, "source figure unavailable; generated placeholder", ha='center', va='center', fontsize=11)
    plt.axis('off')
    savefig(out)
    return out


## 5. Figure 1 — residual manifold overview

In [ ]:
embed = normalize_label_columns(tables.get("residual_universality_embedding"))
if embed is None:
    embed = normalize_label_columns(tables.get("residual_pca_embedding"))

out = FIGURES_DIR / "27_figure_1_residual_manifold_overview.png"
if embed is not None and {"PC1", "PC2"}.issubset(embed.columns):
    plt.figure(figsize=(9, 6))
    if "topology" in embed.columns:
        for topo, sub in embed.groupby("topology"):
            plt.plot(sub["PC1"], sub["PC2"], marker='o', linewidth=2, label=str(topo))
            if "n_modules" in sub.columns:
                for _, r in sub.iterrows():
                    plt.text(r["PC1"], r["PC2"], f"N={int(r['n_modules'])}" if pd.notna(r['n_modules']) else "", fontsize=8)
            elif "N" in sub.columns:
                for _, r in sub.iterrows():
                    plt.text(r["PC1"], r["PC2"], f"N={int(r['N'])}" if pd.notna(r['N']) else "", fontsize=8)
        plt.legend()
    else:
        plt.scatter(embed["PC1"], embed["PC2"], s=80)
    plt.axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.6)
    plt.axvline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.6)
    plt.title("Residual manifold overview")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, alpha=0.3)
    savefig(out)
else:
    copy_or_placeholder(out.name, "Residual manifold overview", keywords=["embedding"] )


## 6. Figure 2 — topology separability

In [ ]:
out = FIGURES_DIR / "27_figure_2_topology_separability.png"
geo = tables.get("residual_geometry_features")
if geo is not None and "topology" in geo.columns:
    numeric = [c for c in geo.columns if c not in {"topology", "n_modules", "N"} and pd.api.types.is_numeric_dtype(geo[c])]
    preferred = [c for c in ["residual_entropy", "residual_asymmetry", "residual_bend_energy", "residual_localization"] if c in numeric]
    cols = preferred or numeric[:4]
    if cols:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.ravel()
        xcol = "n_modules" if "n_modules" in geo.columns else ("N" if "N" in geo.columns else None)
        for ax, col in zip(axes, cols):
            for topo, sub in geo.groupby("topology"):
                if xcol:
                    sub = sub.sort_values(xcol)
                    ax.plot(sub[xcol], sub[col], marker='o', label=str(topo))
                else:
                    ax.bar(str(topo), sub[col].mean())
            ax.set_title(col.replace("_", " "))
            ax.grid(True, alpha=0.3)
        for ax in axes[len(cols):]:
            ax.axis('off')
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='upper center', ncol=min(5, len(labels)))
        fig.suptitle("Residual topology separability", y=1.02, fontsize=16)
        savefig(out)
    else:
        copy_or_placeholder(out.name, "Topology separability", keywords=["separability"])
else:
    copy_or_placeholder(out.name, "Topology separability", keywords=["separability"])


## 7. Figure 3 — universality flow

In [ ]:
out = FIGURES_DIR / "27_figure_3_universality_flow.png"
src = None
for keys in [["universality", "flow"], ["renormalization", "flow"], ["trajectory", "reconstruction"]]:
    src = find_first_existing_figure(keys)
    if src:
        break
if src:
    shutil.copy2(src, out)
    print("copied", src.name, "->", out.name)
else:
    copy_or_placeholder(out.name, "Universality flow", keywords=["flow"])


## 8. Figure 4 — boundary and OOD transfer

In [ ]:
out = FIGURES_DIR / "27_figure_4_boundary_and_ood_transfer.png"
# Build a two-panel if controlled sweep and OOD tables exist; otherwise copy existing boundary/OOD figure.
sweeps = tables.get("controlled_boundary_sweeps")
ood = tables.get("ood_transfer_assignments")
if sweeps is not None or ood is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if sweeps is not None:
        xcol = None
        for c in ["parameter_value", "value", "beta", "p_out", "m", "chord_density"]:
            if c in sweeps.columns:
                xcol = c; break
        ycol = None
        for c in ["relative_confidence", "confidence", "boundary_confidence"]:
            if c in sweeps.columns:
                ycol = c; break
        group = "sweep" if "sweep" in sweeps.columns else ("sweep_name" if "sweep_name" in sweeps.columns else None)
        if xcol and ycol:
            if group:
                for name, sub in sweeps.groupby(group):
                    axes[0].plot(sub[xcol], sub[ycol], marker='o', label=str(name))
                axes[0].legend(fontsize=8)
            else:
                axes[0].plot(sweeps[xcol], sweeps[ycol], marker='o')
            axes[0].set_xlabel(xcol)
            axes[0].set_ylabel(ycol)
        else:
            axes[0].text(0.5,0.5,"controlled sweep table loaded", ha='center')
    else:
        axes[0].text(0.5,0.5,"controlled boundary sweeps unavailable", ha='center')
    axes[0].set_title("Boundary confidence sweeps")
    axes[0].grid(True, alpha=0.3)

    if ood is not None:
        cat = "ood_family" if "ood_family" in ood.columns else ("topology" if "topology" in ood.columns else ("family" if "family" in ood.columns else None))
        val = None
        for c in ["relative_confidence", "confidence", "transfer_similarity", "nearest_distance"]:
            if c in ood.columns:
                val = c; break
        if cat and val:
            agg = ood.groupby(cat)[val].mean().sort_values()
            agg.plot(kind='barh', ax=axes[1])
            axes[1].set_xlabel(val)
        else:
            axes[1].text(0.5,0.5,"OOD transfer table loaded", ha='center')
    else:
        axes[1].text(0.5,0.5,"OOD transfer table unavailable", ha='center')
    axes[1].set_title("OOD transfer summary")
    axes[1].grid(True, alpha=0.3)
    fig.suptitle("Boundary and OOD transfer", y=1.03, fontsize=16)
    savefig(out)
else:
    src = None
    for keys in [["ood", "transfer"], ["boundary", "confidence"]]:
        src = find_first_existing_figure(keys)
        if src: break
    if src:
        shutil.copy2(src, out); print("copied", src.name, "->", out.name)
    else:
        copy_or_placeholder(out.name, "Boundary and OOD transfer")


## 9. Figure 5 — geodesic transport

In [ ]:
out = FIGURES_DIR / "27_figure_5_geodesic_transport.png"
src = None
for keys in [["geodesic", "transport", "paths"], ["transport", "vector"], ["knn", "transport"]]:
    src = find_first_existing_figure(keys)
    if src: break
if src:
    shutil.copy2(src, out)
    print("copied", src.name, "->", out.name)
else:
    copy_or_placeholder(out.name, "Geodesic transport", keywords=["transport"])


## 10. Figure 6 — diffusion persistence

In [ ]:
out = FIGURES_DIR / "27_figure_6_diffusion_persistence.png"
ent = tables.get("diffusion_entropy")
per = tables.get("diffusion_persistence")
if ent is not None or per is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if ent is not None:
        step = "step" if "step" in ent.columns else ("diffusion_step" if "diffusion_step" in ent.columns else ent.columns[0])
        y = "entropy" if "entropy" in ent.columns else None
        group = "topology" if "topology" in ent.columns else None
        if y:
            if group:
                for name, sub in ent.groupby(group):
                    axes[0].plot(sub[step], sub[y], marker='o', label=str(name))
                axes[0].legend(fontsize=8)
            else:
                axes[0].plot(ent[step], ent[y], marker='o')
            axes[0].set_xlabel(step); axes[0].set_ylabel(y)
        else:
            axes[0].text(0.5,0.5,"entropy table loaded", ha='center')
    else:
        axes[0].text(0.5,0.5,"entropy curves unavailable", ha='center')
    axes[0].set_title("Diffusion entropy")
    axes[0].grid(True, alpha=0.3)

    if per is not None:
        step = "step" if "step" in per.columns else ("diffusion_step" if "diffusion_step" in per.columns else per.columns[0])
        y = None
        for c in ["persistence", "universality_persistence", "mass_retained"]:
            if c in per.columns:
                y = c; break
        group = "topology" if "topology" in per.columns else None
        if y:
            if group:
                for name, sub in per.groupby(group):
                    axes[1].plot(sub[step], sub[y], marker='o', label=str(name))
                axes[1].legend(fontsize=8)
            else:
                axes[1].plot(per[step], per[y], marker='o')
            axes[1].set_xlabel(step); axes[1].set_ylabel(y)
        else:
            axes[1].text(0.5,0.5,"persistence table loaded", ha='center')
    else:
        axes[1].text(0.5,0.5,"persistence curves unavailable", ha='center')
    axes[1].set_title("Universality persistence")
    axes[1].grid(True, alpha=0.3)
    fig.suptitle("Residual diffusion persistence", y=1.03, fontsize=16)
    savefig(out)
else:
    src = None
    for keys in [["entropy", "persistence"], ["persistence", "decay"], ["diffusion"]]:
        src = find_first_existing_figure(keys)
        if src: break
    if src:
        shutil.copy2(src, out); print("copied", src.name, "->", out.name)
    else:
        copy_or_placeholder(out.name, "Diffusion persistence")


## 11. Figure 7 — spectral / phase persistence supplement

In [ ]:
out = FIGURES_DIR / "27_figure_7_spectral_phase_supplement.png"
phase = tables.get("diffusion_phase_grid")
spec = tables.get("laplacian_spectrum")
if phase is not None or spec is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if phase is not None:
        x = next((c for c in ["alpha", "diffusion_alpha", "diffusion step size alpha"] if c in phase.columns), None)
        y = next((c for c in ["k", "knn_k", "graph_sparsity_k"] if c in phase.columns), None)
        z = next((c for c in ["mean_persistence", "persistence", "universality_persistence"] if c in phase.columns), None)
        if x and y and z:
            piv = phase.pivot_table(index=y, columns=x, values=z, aggfunc='mean')
            im = axes[0].imshow(piv.values, aspect='auto', origin='lower')
            axes[0].set_xticks(range(len(piv.columns))); axes[0].set_xticklabels([str(c) for c in piv.columns], rotation=45, ha='right')
            axes[0].set_yticks(range(len(piv.index))); axes[0].set_yticklabels([str(c) for c in piv.index])
            axes[0].set_xlabel(x); axes[0].set_ylabel(y)
            fig.colorbar(im, ax=axes[0], label=z)
        else:
            axes[0].text(0.5,0.5,"phase grid loaded", ha='center')
    else:
        axes[0].text(0.5,0.5,"phase grid unavailable", ha='center')
    axes[0].set_title("Persistence phase grid")

    if spec is not None:
        mode = "mode" if "mode" in spec.columns else spec.columns[0]
        eig = "eigenvalue" if "eigenvalue" in spec.columns else ("lambda" if "lambda" in spec.columns else None)
        if eig:
            axes[1].plot(spec[mode], spec[eig], marker='o')
            axes[1].set_xlabel(mode); axes[1].set_ylabel(eig)
        else:
            axes[1].text(0.5,0.5,"spectrum table loaded", ha='center')
    else:
        axes[1].text(0.5,0.5,"spectrum unavailable", ha='center')
    axes[1].set_title("Laplacian spectrum")
    axes[1].grid(True, alpha=0.3)
    fig.suptitle("Spectral and phase persistence supplement", y=1.03, fontsize=16)
    savefig(out)
else:
    src = None
    for keys in [["phase", "diagram"], ["laplacian", "spectrum"], ["spectral"]]:
        src = find_first_existing_figure(keys)
        if src: break
    if src:
        shutil.copy2(src, out); print("copied", src.name, "->", out.name)
    else:
        copy_or_placeholder(out.name, "Spectral phase supplement")


## 12. Draft figure captions

In [ ]:
captions = {
    "27_figure_1_residual_manifold_overview.png": "Figure 1. Residual manifold overview. Topology-size points are embedded in a shared residual coordinate system, showing how residual geometry separates and connects graph families across scale.",
    "27_figure_2_topology_separability.png": "Figure 2. Residual topology separability. Residual feature trends summarize entropy, asymmetry, bend energy, and localization across graph size and topology.",
    "27_figure_3_universality_flow.png": "Figure 3. Universality flow. Residual trajectories across scale trace topology-dependent flow directions and fixed-structure tendencies in the learned manifold.",
    "27_figure_4_boundary_and_ood_transfer.png": "Figure 4. Boundary confidence and out-of-distribution transfer. Controlled boundary sweeps and OOD assignments test whether residual universality regions generalize beyond the original topology set.",
    "27_figure_5_geodesic_transport.png": "Figure 5. Geodesic transport over the residual manifold. Transport paths reveal bridgeable regions, bottlenecks, and high-cost transitions between topology families.",
    "27_figure_6_diffusion_persistence.png": "Figure 6. Residual diffusion persistence. Entropy growth and mass-retention curves measure whether topology-localized residual information persists under repeated diffusion.",
    "27_figure_7_spectral_phase_supplement.png": "Figure 7. Spectral and phase persistence supplement. Laplacian modes and diffusion phase grids summarize manifold-scale stability, mixing, and collapse thresholds.",
}

caption_path = RESULTS_DIR / "27_figure_caption_drafts.md"
with open(caption_path, "w") as f:
    f.write("# Figure caption drafts\n\n")
    for name, cap in captions.items():
        f.write(f"## {name}\n\n{cap}\n\n")
print("wrote", caption_path)
print(caption_path.read_text())


## 13. Build paper-ready export pack

In [ ]:
manifest = {
    "notebook": "27_paper_figures_and_results_pack.ipynb",
    "repo_root": str(REPO_ROOT),
    "created_outputs": {
        "figures": sorted([p.name for p in FIGURES_DIR.glob("27_*.png")]),
        "results": sorted([p.name for p in RESULTS_DIR.glob("27_*")]),
        "exports": [],
    },
    "loaded_tables": sorted(loaded_tables.keys()),
    "core_files_checked": core_files,
}

manifest_path = EXPORTS_DIR / "27_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / "27_paper_figures_and_results_pack_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FIGURES_DIR.glob("27_*.png")):
        z.write(p, arcname=f"figures/{p.name}")
    for p in sorted(RESULTS_DIR.glob("27_*")):
        z.write(p, arcname=f"results/{p.name}")
    z.write(manifest_path, arcname="exports/27_manifest.json")

manifest["created_outputs"]["exports"] = [manifest_path.name, zip_path.name]
manifest_path.write_text(json.dumps(manifest, indent=2))

print("Wrote:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / 1e6, 3))


## 14. Optional Colab download

Run this cell in Colab to download the export zip.

In [ ]:
# Optional Colab download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Colab download skipped. Download manually from:", zip_path)
    print("Reason:", e)
